# Featuer Engneering

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

df_clean = pd.read_csv(
    PROJECT_ROOT / "data" / "cleaned" / "iot23_cleaned.csv"
)
df_features = df_clean.copy()

In [ ]:

df_features["total_pkts"] = (
    df_features["orig_pkts"]
    + df_features["resp_pkts"]
)

df_features["total_ip_bytes"] = (
    df_features["orig_ip_bytes"]
    + df_features["resp_ip_bytes"]
)

# A zero denominator produces a missing value, not infinity.
df_features["orig_avg_pkt_size"] = (
    df_features["orig_ip_bytes"]
    / df_features["orig_pkts"].replace(0, np.nan)
)

df_features["resp_avg_pkt_size"] = (
    df_features["resp_ip_bytes"]
    / df_features["resp_pkts"].replace(0, np.nan)
)

df_features["resp_pkt_share"] = (
    df_features["resp_pkts"]
    / df_features["total_pkts"].replace(0, np.nan)
)

df_features["no_response_pkts"] = (
    df_features["resp_pkts"].eq(0).astype("int8")
)

# Only calculate rates for known, positive durations.
valid_duration = df_features["duration"].where(
    df_features["duration"] > 0
)

df_features["pkts_per_second"] = (
    df_features["total_pkts"] / valid_duration
)

In [9]:
for column in ["duration", "orig_bytes", "resp_bytes", "service"]:
    df_features[f"{column}_missing"] = (
        df_features[column].isna().astype("int8")
    )

In [10]:
df_features = df_features.drop(
    columns=["local_orig", "local_resp", "tunnel_parents"]
)

In [11]:
excluded_from_features = [
    "uid",
    "label",
    "detailed_label",
    "target",
]

In [12]:
df_features

,ts,uid,id_orig_h,id_orig_p,id_resp_h,id_resp_p,proto,service,duration,orig_bytes,...,total_ip_bytes,orig_avg_pkt_size,resp_avg_pkt_size,resp_pkt_share,no_response_pkts,pkts_per_second,duration_missing,orig_bytes_missing,resp_bytes_missing,service_missing
0,1.545404e+09,CrDn63WjJEmrWGjqf,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,3.139211,0.0,...,180,60.000000,NaN,0.000000,1,0.955654,0,0,0,1
1,1.545404e+09,CY9lJW3gh1Eje4usP6,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,60,60.000000,NaN,0.000000,1,NaN,1,1,1,1
2,1.545404e+09,CcFXLynukEDnUlvgl,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,60,60.000000,NaN,0.000000,1,NaN,1,1,1,1
3,1.545404e+09,CDrkrSobGYxHhYfth,192.168.1.195,41040,185.244.25.235,80,tcp,http,1.477656,149.0,...,144569,58.776596,1448.375000,0.505263,0,128.582025,0,0,0,0
4,1.545404e+09,CTWZQf2oJSvq6zmPAc,192.168.1.195,41042,185.244.25.235,80,tcp,NaN,3.147116,0.0,...,180,60.000000,NaN,0.000000,1,0.953254,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23140,1.545490e+09,C2F17zSUnGOcWzBa7,192.168.1.195,57110,185.244.25.235,6667,tcp,irc,32.840994,62.0,...,1023,62.000000,98.166667,0.461538,0,0.395847,0,0,0,0
23141,1.545490e+09,C93P4z4k5IRJD1rXJg,192.168.1.195,57092,185.244.25.235,6667,tcp,irc,36.290833,62.0,...,1238,60.600000,90.285714,0.411765,0,0.468438,0,0,0,0
23142,1.545490e+09,CXLZ3A2QY5E8weqpDk,192.168.1.195,123,147.251.48.140,123,udp,NaN,NaN,NaN,...,76,76.000000,NaN,0.000000,1,NaN,1,1,1,1
23143,1.545490e+09,CuXpFN3fWesWBXUhq1,192.168.1.195,123,82.113.53.40,123,udp,NaN,NaN,NaN,...,76,76.000000,NaN,0.000000,1,NaN,1,1,1,1
